<a href="https://colab.research.google.com/github/HussainSarwar09/finsite-ai/blob/main/app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile app_final.py
import streamlit as st
import os
import pandas as pd
from huggingface_hub import InferenceClient
from fin_graph import create_agent
from fin_tools import init_vector_db

st.set_page_config(page_title="FinInsight + Qdrant UI", layout="wide")

# --- 1. SETUP SYSTEMS ---
@st.cache_resource
def setup_systems():
    token = os.getenv('MY_HF_TOKEN')
    client = InferenceClient(model="meta-llama/Meta-Llama-3-8B-Instruct", token=token)
    qdrant, embed_model = init_vector_db()
    agent = create_agent(client, embed_model, qdrant)
    return agent, qdrant

agent, q_client = setup_systems()

# --- 2. QDRANT IN-MEMORY UI (SIDEBAR) ---
with st.sidebar:
    st.title("🛠️ Qdrant Memory UI")
    if st.button("🔄 Refresh Vector DB View"):
        # Use scroll to view in-memory points
        points, _ = q_client.scroll(collection_name="fin_mem", limit=20, with_payload=True)
        if points:
            data = [{"ID": p.id, **p.payload} for p in points]
            st.table(pd.DataFrame(data))
        else:
            st.warning("Memory is empty. Fetch news for a stock first!")
    st.markdown("---")
    st.info("In-memory data persists as long as the Colab session is active.")

# --- 3. CHAT INTERFACE ---
st.title("💹 FinInsight AI Copilot")
if "messages" not in st.session_state:
    st.session_state.messages = []

for m in st.session_state.messages:
    with st.chat_message(m["role"]): st.markdown(m["content"])

if prompt := st.chat_input("Ex: Analyze Apple price and news..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"): st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Agent is researching..."):
            res = agent.invoke({"query": prompt, "outs": []})
            st.markdown(res["ans"])
            st.session_state.messages.append({"role": "assistant", "content": res["ans"]})